In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
file_path = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\synthetic_imu_data_01_withBias_1h_stat_low_high.txt"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Shape: (180000, 18)
Columns:
['IMU_low_ax', 'IMU_low_ay', 'IMU_low_az', 'IMU_low_gx', 'IMU_low_gy', 'IMU_low_gz', 'IMU_low_mx', 'IMU_low_my', 'IMU_low_mz', 'IMU_high_ax', 'IMU_high_ay', 'IMU_high_az', 'IMU_high_gx', 'IMU_high_gy', 'IMU_high_gz', 'IMU_high_mx', 'IMU_high_my', 'IMU_high_mz']

First 5 rows:


,IMU_low_ax,IMU_low_ay,IMU_low_az,IMU_low_gx,IMU_low_gy,IMU_low_gz,IMU_low_mx,IMU_low_my,IMU_low_mz,IMU_high_ax,IMU_high_ay,IMU_high_az,IMU_high_gx,IMU_high_gy,IMU_high_gz,IMU_high_mx,IMU_high_my,IMU_high_mz
0,-1.8023,2.1978,10.815,0.010248,-0.011721,0.017139,22.808,-4.6621,-10.561,-0.002078,0.001589,9.8102,0.000031,-0.000003,0.000005,27.471,-2.3455,-16.174
1,-1.8037,2.1922,10.833,0.010432,-0.010397,0.018632,22.508,-4.5100,-11.318,0.000836,0.001883,9.8069,0.000132,-0.000184,-0.000133,27.561,-2.4420,-16.168
2,-1.8195,2.2347,10.814,0.009880,-0.008280,0.019564,22.512,-4.2728,-11.272,0.004230,-0.003740,9.8137,-0.000034,0.000056,0.000241,27.640,-2.4091,-16.138
3,-1.8085,2.2029,10.826,0.009108,-0.010034,0.016525,22.413,-4.3540,-11.129,0.004189,0.001732,9.8112,0.000004,-0.000167,-0.000158,27.396,-2.5745,-16.217
4,-1.8043,2.1952,10.818,0.011408,-0.010205,0.017245,22.743,-4.4115,-11.003,-0.000704,0.000900,9.8122,0.000143,0.000330,0.000031,27.540,-2.4004,-16.305


In [3]:
for axis in ["x", "y", "z"]:
    df[f"error_{axis}"] = (
        df[f"IMU_low_a{axis}"] -
        df[f"IMU_high_a{axis}"]
    )

print(df[["error_x", "error_y", "error_z"]].describe())

             error_x        error_y        error_z
count  180000.000000  180000.000000  180000.000000
mean       -1.867105       2.227332       0.886849
std         0.053651       0.051642       0.083363
min        -2.058673       2.083275       0.692500
25%        -1.888061       2.187679       0.826400
50%        -1.857460       2.220315       0.860200
75%        -1.832643       2.270846       0.946000
max        -1.715456       2.385355       1.137100


In [4]:
for axis in ["x", "y", "z"]:
    df[f"noise_{axis}"] = (
        df[f"error_{axis}"] -
        df[f"error_{axis}"].mean()
    )

print("Mean total error:")
print(df[["error_x", "error_y", "error_z"]].mean())

print("\nMean-removed residual:")
print(df[["noise_x", "noise_y", "noise_z"]].mean())

Mean total error:
error_x   -1.867105
error_y    2.227332
error_z    0.886849
dtype: float64

Mean-removed residual:
noise_x   -8.589672e-17
noise_y    1.288451e-16
noise_z    1.515825e-16
dtype: float64


In [5]:
feature_cols = [
    "IMU_low_ax",
    "IMU_low_ay",
    "IMU_low_az"
]

target_cols = [
    "noise_x",
    "noise_y",
    "noise_z"
]

window = 10

data = df[feature_cols + target_cols].dropna()

X = []
Y = []

for i in range(window, len(data)):
    X.append(
        data[feature_cols].iloc[i-window:i].values.flatten()
    )
    Y.append(
        data[target_cols].iloc[i].values
    )

X = np.array(X)
Y = np.array(Y)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (179990, 30)
Y shape: (179990, 3)


In [6]:
split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

Y_train = Y[:split]
Y_test = Y[split:]

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (143992, 30)
Testing : (35998, 30)


In [7]:
model = Ridge(alpha=1.0)

model.fit(X_train, Y_train)

Y_pred = model.predict(X_test)

In [8]:
for i, axis in enumerate(["X", "Y", "Z"]):

    baseline_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            np.zeros(len(Y_test))
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            Y_pred[:, i]
        )
    )

    mae = mean_absolute_error(
        Y_test[:, i],
        Y_pred[:, i]
    )

    improvement = (
        1 - ml_rmse / baseline_rmse
    ) * 100

    print(f"{axis}-axis")
    print(f"Baseline RMSE : {baseline_rmse:.6f}")
    print(f"ML RMSE       : {ml_rmse:.6f}")
    print(f"ML MAE        : {mae:.6f}")
    print(f"Improvement   : {improvement:.2f}%")
    print()

X-axis
Baseline RMSE : 0.092195
ML RMSE       : 0.017306
ML MAE        : 0.013812
Improvement   : 81.23%

Y-axis
Baseline RMSE : 0.049022
ML RMSE       : 0.016922
ML MAE        : 0.013505
Improvement   : 65.48%

Z-axis
Baseline RMSE : 0.085700
ML RMSE       : 0.016984
ML MAE        : 0.013571
Improvement   : 80.18%



In [9]:
dynamic_file = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\synthetic_imu_data_04_withBias_1h_dyn_low_high.txt"

df_dyn = pd.read_csv(dynamic_file)

print("Shape:", df_dyn.shape)
print(df_dyn.columns.tolist())
display(df_dyn.head())

Shape: (360000, 18)
['IMU_low_ax', 'IMU_low_ay', 'IMU_low_az', 'IMU_low_gx', 'IMU_low_gy', 'IMU_low_gz', 'IMU_low_mx', 'IMU_low_my', 'IMU_low_mz', 'IMU_high_ax', 'IMU_high_ay', 'IMU_high_az', 'IMU_high_gx', 'IMU_high_gy', 'IMU_high_gz', 'IMU_high_mx', 'IMU_high_my', 'IMU_high_mz']


,IMU_low_ax,IMU_low_ay,IMU_low_az,IMU_low_gx,IMU_low_gy,IMU_low_gz,IMU_low_mx,IMU_low_my,IMU_low_mz,IMU_high_ax,IMU_high_ay,IMU_high_az,IMU_high_gx,IMU_high_gy,IMU_high_gz,IMU_high_mx,IMU_high_my,IMU_high_mz
0,-1.7981,2.1903,10.815,0.75025,0.49180,0.57149,22.547,-4.4095,-11.219,0.000621,0.003010,9.8037,0.73965,0.49991,0.55010,27.615,-2.3756,-16.142
1,-1.8374,2.2827,10.793,0.75066,0.49419,0.56831,23.123,-4.5032,-10.969,-0.049962,0.070317,9.8046,0.73979,0.50085,0.55075,27.652,-2.8195,-15.988
2,-1.8993,2.3255,10.848,0.75022,0.48966,0.57192,22.727,-4.9531,-10.583,-0.098505,0.142820,9.8063,0.74010,0.50099,0.55188,27.778,-2.8591,-15.728
3,-1.9321,2.4286,10.816,0.74815,0.48952,0.57286,22.629,-5.6227,-10.570,-0.147670,0.219620,9.8060,0.74059,0.50205,0.55253,27.719,-3.2480,-15.641
4,-2.0081,2.4668,10.806,0.74965,0.49153,0.57139,23.038,-5.4410,-10.319,-0.196950,0.295930,9.7984,0.74150,0.50250,0.55369,27.569,-3.4158,-15.412


In [10]:
for axis in ["x", "y", "z"]:
    df_dyn[f"error_{axis}"] = (
        df_dyn[f"IMU_low_a{axis}"] -
        df_dyn[f"IMU_high_a{axis}"]
    )

print(
    df_dyn[
        ["error_x", "error_y", "error_z"]
    ].describe()
)

             error_x        error_y        error_z
count  360000.000000  360000.000000  360000.000000
mean       -2.005781       2.106845       0.965404
std         0.167525       0.179763       0.168619
min        -2.438300       1.661800       0.466920
25%        -2.141368       1.965368       0.831000
50%        -2.008600       2.106100       0.964700
75%        -1.868845       2.244700       1.102000
max        -1.584390       2.649480       1.374440


In [11]:
accel = df_dyn[
    [
        "IMU_low_ax",
        "IMU_low_ay",
        "IMU_low_az"
    ]
].values

errors = df_dyn[
    [
        "error_x",
        "error_y",
        "error_z"
    ]
].values

window = 10

X_dyn = []
Y_dyn = []

for i in range(window, len(df_dyn)):
    X_dyn.append(
        accel[i-window:i].flatten()
    )
    Y_dyn.append(
        errors[i]
    )

X_dyn = np.asarray(X_dyn)
Y_dyn = np.asarray(Y_dyn)

print("X shape:", X_dyn.shape)
print("Y shape:", Y_dyn.shape)

X shape: (359990, 30)
Y shape: (359990, 3)


In [12]:
split = int(0.8 * len(X_dyn))

X_train_dyn = X_dyn[:split]
X_test_dyn = X_dyn[split:]

Y_train_dyn = Y_dyn[:split]
Y_test_dyn = Y_dyn[split:]

print("Training:", X_train_dyn.shape)
print("Testing :", X_test_dyn.shape)

Training: (287992, 30)
Testing : (71998, 30)


In [13]:
scaler = StandardScaler()

X_train_dyn_scaled = scaler.fit_transform(X_train_dyn)
X_test_dyn_scaled = scaler.transform(X_test_dyn)

model_dyn = Ridge(alpha=1.0)

model_dyn.fit(
    X_train_dyn_scaled,
    Y_train_dyn
)

Y_pred_dyn = model_dyn.predict(
    X_test_dyn_scaled
)

In [14]:
baseline_pred = np.tile(
    Y_train_dyn.mean(axis=0),
    (len(Y_test_dyn), 1)
)

for i, axis in enumerate(["X", "Y", "Z"]):

    baseline_rmse = np.sqrt(
        mean_squared_error(
            Y_test_dyn[:, i],
            baseline_pred[:, i]
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            Y_test_dyn[:, i],
            Y_pred_dyn[:, i]
        )
    )

    ml_mae = mean_absolute_error(
        Y_test_dyn[:, i],
        Y_pred_dyn[:, i]
    )

    improvement = (
        1 - ml_rmse / baseline_rmse
    ) * 100

    print(f"{axis}-axis")
    print(f"Baseline RMSE : {baseline_rmse:.6f}")
    print(f"ML RMSE       : {ml_rmse:.6f}")
    print(f"ML MAE        : {ml_mae:.6f}")
    print(f"Improvement   : {improvement:.2f}%")
    print()

X-axis
Baseline RMSE : 0.187242
ML RMSE       : 0.098545
ML MAE        : 0.089423
Improvement   : 47.37%

Y-axis
Baseline RMSE : 0.181466
ML RMSE       : 0.084491
ML MAE        : 0.066963
Improvement   : 53.44%

Z-axis
Baseline RMSE : 0.191490
ML RMSE       : 0.110773
ML MAE        : 0.094108
Improvement   : 42.15%



In [15]:
X_instant = df_dyn[
    [
        "IMU_low_ax",
        "IMU_low_ay",
        "IMU_low_az"
    ]
].values

Y_instant = df_dyn[
    [
        "error_x",
        "error_y",
        "error_z"
    ]
].values

split = int(0.8 * len(X_instant))

Xtr = X_instant[:split]
Xte = X_instant[split:]

Ytr = Y_instant[:split]
Yte = Y_instant[split:]

scaler_instant = StandardScaler()

Xtr_s = scaler_instant.fit_transform(Xtr)
Xte_s = scaler_instant.transform(Xte)

model_instant = Ridge(alpha=1.0)

model_instant.fit(Xtr_s, Ytr)

Ypred_instant = model_instant.predict(Xte_s)

for i, axis in enumerate(["X", "Y", "Z"]):

    rmse = np.sqrt(
        mean_squared_error(
            Yte[:, i],
            Ypred_instant[:, i]
        )
    )

    mae = mean_absolute_error(
        Yte[:, i],
        Ypred_instant[:, i]
    )

    print(f"{axis}-axis")
    print(f"Instantaneous RMSE: {rmse:.6f}")
    print(f"Instantaneous MAE : {mae:.6f}")

X-axis
Instantaneous RMSE: 0.098545
Instantaneous MAE : 0.089417
Y-axis
Instantaneous RMSE: 0.084489
Instantaneous MAE : 0.066965
Z-axis
Instantaneous RMSE: 0.110773
Instantaneous MAE : 0.094106
